# 08 · Recursion with matrices and vectors

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/08-recursion-with-matrices.ipynb)

*Part IV · demo · 10 min*

> 🇪🇸 **Recursión con matrices y vectores** — Aplicar una misma matriz una y otra vez: Fibonacci, autovectores y un pronóstico real.

Apply one matrix again and again: Fibonacci, eigenvectors, and a real forecast.

## What you will be able to do

- Write a recurrence as repeated multiplication by one matrix.
- Find the dominant eigenvector by power iteration, and check it against `np.linalg.eig`.
- Fit an autoregressive model with the pseudoinverse and feed its own output back in.
- Recognise that structure as the skeleton of a recurrent neural network.

## Setup

Run this first. It installs and imports everything this notebook needs, and nothing else.

> 🇪🇸 Ejecuta esto primero: instala e importa todo lo que este cuaderno necesita.

In [ ]:
import numpy as np
import pandas as pd

FLIGHTS = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/flights.csv"
flights = pd.read_csv(FLIGHTS)

rng = np.random.default_rng(0)
print(flights.shape)                       # (144, 3) — 144 real months, 1949-1960

## Recursion means defining something in terms of itself

> 🇪🇸 La recursión consiste en definir algo en términos de sí mismo. Con
> matrices, esto se convierte en aplicar la misma matriz una y otra vez.

With matrices this becomes: **apply the same matrix again and again.** Three
examples, increasing in usefulness.

This is a demo — read it, run it, ask about it. The exercises at the end are
short.

### 1. Fibonacci as repeated matrix multiplication

The rule `f(n) = f(n-1) + f(n-2)` is one matrix applied repeatedly.

In [ ]:
F = np.array([[1, 1], [1, 0]])
v = np.array([1, 0])
for _ in range(10):
    v = F @ v
print(v[1])                                  # 55
print(np.linalg.matrix_power(F, 10)[0, 1])   # 55 — same answer, one step

### 2. Power iteration — recursion that finds an eigenvector

Multiply any starting vector by `A` repeatedly, rescaling each time. It
converges to the eigenvector with the largest eigenvalue (Chapter 2 §2.7).

In [ ]:
A = np.array([[4., 1.], [2., 3.]])
x = rng.standard_normal(2); x /= np.linalg.norm(x)
for _ in range(50):
    x = A @ x
    x /= np.linalg.norm(x)

print(x @ A @ x)                    # 5.000000
print(np.linalg.eig(A)[0].max())    # 5.000000 — identical

In [ ]:
import matplotlib.pyplot as plt

xv = rng.standard_normal(2); xv /= np.linalg.norm(xv)
checkpoints = {}
for step in range(1, 51):
    xv = A @ xv
    xv /= np.linalg.norm(xv)
    if step in (1, 2, 5, 10, 50):
        checkpoints[step] = xv.copy()

eigvals, eigvecs = np.linalg.eig(A)
dominant = eigvecs[:, np.argmax(eigvals)].real
dominant /= np.linalg.norm(dominant)

fig, ax = plt.subplots(figsize=(4, 4))
theta = np.linspace(0, 2 * np.pi, 200)
ax.plot(np.cos(theta), np.sin(theta), color="lightgray", linewidth=1)
for step, v in checkpoints.items():
    ax.annotate("", xy=v, xytext=(0, 0), arrowprops=dict(
        arrowstyle="->", color="#4C72B0", alpha=0.3 + 0.7 * step / 50))
    ax.text(v[0] * 1.15, v[1] * 1.15, str(step), fontsize=8, ha="center")
for sign in (1, -1):
    ax.annotate("", xy=sign * dominant, xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", color="#C44E52", linewidth=2))
ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.3, 1.3); ax.set_aspect("equal")
ax.set_title("power iteration converges to the eigenvector\n(red = the true dominant eigenvector, both signs)")
plt.tight_layout()
plt.show()

This is how PageRank ranks web pages, and it is why eigenvectors matter far
beyond Chapter 2: **repeated application of a matrix converges to its dominant
eigenvector.**

### 3. Recursion on real data — forecasting airline traffic

This combines recursion with the pseudoinverse from section 07. We fit a model
that predicts each month from the previous 12, then apply it *to its own output*
to forecast forward.

In [ ]:
y = flights['passengers'].to_numpy(float)     # 144 real months, 1949-1960
p = 12
rows = np.array([y[i:i+p] for i in range(len(y) - p)])
X = np.column_stack([np.ones(len(rows)), rows])
w = np.linalg.pinv(X) @ y[p:]                 # least squares, exactly as in section 07
print(X.shape)                                # (132, 13) — 132 training windows

history = list(y[-p:])
for _ in range(12):                           # recursion: feed predictions back in
    nxt = w[0] + np.dot(w[1:], history[-p:])
    history.append(nxt)

print(np.round(history[-12:], 1))
# [465.2 429.1 455.1 491.0 527.8 589.4 679.7 661.3 575.3 509.5 438.6 470.7]

The forecast above extrapolates 12 months **past the end of the dataset**, so
there is nothing to check it against. To see the forecast next to real numbers,
hold out the last 12 months, fit on everything before them, and forecast those
same 12 months back.

> 🇪🇸 El pronóstico anterior se extiende 12 meses **más allá del final de los
> datos**, así que no hay nada real con qué compararlo. Para ver el pronóstico
> junto a números reales, se retienen los últimos 12 meses, se ajusta con todo
> lo anterior, y se pronostican esos mismos 12 meses.

In [ ]:
y_train, y_test = y[:-12], y[-12:]
rows_tr = np.array([y_train[i:i+p] for i in range(len(y_train) - p)])
X_tr = np.column_stack([np.ones(len(rows_tr)), rows_tr])
w_tr = np.linalg.pinv(X_tr) @ y_train[p:]

hist_tr = list(y_train[-p:])
for _ in range(12):
    hist_tr.append(w_tr[0] + np.dot(w_tr[1:], hist_tr[-p:]))
forecast_holdout = np.array(hist_tr[-12:])

import matplotlib.pyplot as plt
months = np.arange(1, 13)
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(months, y_test, marker="o", label="actual", color="#4C72B0")
ax.plot(months, forecast_holdout, marker="o", label="forecast", color="#C44E52")
ax.set_xlabel("month (held out, never seen while fitting)")
ax.set_ylabel("passengers")
ax.set_title("forecast vs. actual — last 12 months held out")
ax.legend()
plt.tight_layout()
plt.show()

mape = (np.abs(forecast_holdout - y_test) / y_test).mean()
print(f"mean absolute percentage error: {mape:.1%}")

The forecast reproduces the seasonal shape of real air travel — low in winter,
peaking in summer — because the model learned it from 132 real training windows.

**This is exactly the structure of a recurrent neural network**: a hidden state,
updated by the same weights at every step.

In [ ]:
# The same shape, with a nonlinearity. W and U are the SAME at every step —
# that is the recursion.
W = rng.standard_normal((4, 4)) * 0.5
U = rng.standard_normal((4, 3)) * 0.5
xs = rng.standard_normal((6, 3))        # a sequence of 6 inputs, 3 features each

h = np.zeros(4)
for t in range(6):
    h = np.tanh(W @ h + U @ xs[t])
print(np.round(h, 3))

## Exercise 1 — the forecast, and what breaks it

> 🇪🇸 El pronóstico, y qué lo rompe.

In [ ]:
# TODO 1: Change the window length p from 12 to 3 and re-run the forecast.
#         The seasonal shape disappears. Why? What does p = 12 encode about
#         this particular dataset that p = 3 cannot?

# TODO 2: Forecast 60 months ahead instead of 12. Plot it if you can. Recursive
#         forecasting feeds predictions back in as if they were observations —
#         what does that do to the error over a long horizon?

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
def forecast(y, p, steps):
    rows = np.array([y[i:i+p] for i in range(len(y) - p)])
    X = np.column_stack([np.ones(len(rows)), rows])
    w = np.linalg.pinv(X) @ y[p:]
    hist = list(y[-p:])
    for _ in range(steps):
        hist.append(w[0] + np.dot(w[1:], hist[-p:]))
    return np.array(hist[-steps:])

print(np.round(forecast(y, 12, 12), 1))   # seasonal: winter low, summer peak
print(np.round(forecast(y, 3, 12), 1))    # smooth, seasonality gone

# p = 12 encodes ONE YEAR. The model can see the same month a year earlier, so
# seasonality is available to it as a linear term. With p = 3 it can only see a
# local trend, and a linear model has no way to invent a yearly cycle.

print(np.round(forecast(y, 12, 60)[-6:], 1))
# Errors compound: every predicted month becomes an input to the next
# prediction, so mistakes feed on themselves. Recursive forecasts are trustworthy
# for a short horizon and decorative for a long one.

## Exercise 2 — power iteration by hand

> 🇪🇸 Iteración de potencias, paso a paso.

In [ ]:
# TODO 3: Run power iteration on A = [[4., 1.], [2., 3.]] but print the
#         estimate after 1, 2, 5, 10 and 50 steps. How fast does it converge?
#         Try a second matrix whose two eigenvalues are close together
#         (e.g. [[4., 1.], [0., 3.9]]). What changes, and why?

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
def power_iterate(M, steps, seed=0):
    v = np.random.default_rng(seed).standard_normal(M.shape[0])
    v /= np.linalg.norm(v)
    out = {}
    for k in range(1, max(steps) + 1):
        v = M @ v
        v /= np.linalg.norm(v)
        if k in steps:
            out[k] = round(float(v @ M @ v), 4)
    return out

A  = np.array([[4., 1.], [2., 3.]])         # eigenvalues 5 and 2
A2 = np.array([[4., 1.], [0., 3.9]])        # eigenvalues 4 and 3.9

print(power_iterate(A,  [1, 2, 5, 10, 50]))
print(power_iterate(A2, [1, 2, 5, 10, 50]))
print(np.linalg.eigvals(A), np.linalg.eigvals(A2))

# Convergence speed is set by the RATIO of the two largest eigenvalues. For A
# that ratio is 2/5, so each step shrinks the error to 40% of itself and ten
# steps are plenty. For A2 it is 3.9/4 = 0.975, and after 50 steps it is still
# arriving. Power iteration is fast exactly when one direction dominates.

---

## Done with this section

Next up: **09 · Convolution and deconvolution** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/09-convolution-and-deconvolution.ipynb).

[← Back to the workshop site](https://project-delphi.github.io/tensors-workshop/) · [All notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)